# G1 unified inverse reconstruction
One shared network per configuration across Poisson, heat and wave on eight fixed qualified geometry instances. **Preview is the default.** Setup → dataset → preflight → one sequential training call → independent evaluation. No older notebook is required.

## 1. Setup and persistence
Open this file directly from GitHub in Colab. Configure the repository URL below and preferably pin `REPO_REF` to an immutable commit. Private repositories use an external credential helper or SSH agent; never put tokens in cells, URLs or logs. Local CPU/MPS uses the installed checkout without cloning or importing Drive.

In [ ]:
# Setup configuration — used only in Colab. Local sessions need an installed project.
REPO_URL = "SET_YOUR_GITHUB_REPOSITORY_URL"
REPO_REF = "main"  # Prefer an immutable full commit SHA or release tag for research.
PROJECT_DIR = "/content/pde-inverse-research"

USE_GOOGLE_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/PDEInverseResearch"
LOCAL_OUTPUT_DIR = "/content/pde-inverse-outputs"

# Geometry installation extra required for G1_smooth_v1.
ENABLE_GEOMETRY = True


In [ ]:
# Automatic setup — advanced implementation (collapsed). Run this cell normally.
# Stdlib bootstrap is embedded because pdeep is absent in a fresh Colab runtime.
import sys, json
from pathlib import Path
try:
    from IPython import get_ipython
    _shell = get_ipython()
except ImportError:
    _shell = None
IS_COLAB = "google.colab" in sys.modules or type(_shell).__module__.startswith("google.colab")

def ensure_checkout(repo_url, repo_ref, project_dir, *, git_env=None):
    """Clone only absent destinations; existing worktrees are strictly read-only.

    Supports token-free HTTPS and Git-over-SSH URLs. External credential helpers
    or SSH agents handle private authentication. Raw Git output stays in memory.
    """
    import os, re, subprocess, tempfile
    from pathlib import Path
    from urllib.parse import urlsplit
    import sys
    if sys.version_info < (3, 10):
        raise RuntimeError('PDEep requires Python >=3.10; select a supported runtime before setup.')
    def identity(value):
        if not isinstance(value,str) or not value or any(c in value for c in '\n\r\0'):
            raise ValueError('Configure a token-free repository URL; credentials must not be embedded.')
        if value.startswith('git@'):
            match=re.fullmatch(r'git@([A-Za-z0-9.-]+):([A-Za-z0-9_./-]+)',value)
            if not match:raise ValueError('Use git@HOST:OWNER/REPO.git with an external SSH agent.')
            host,path=match.groups()
        else:
            url=urlsplit(value)
            if url.scheme not in {'https','ssh'} or not url.hostname or url.password or url.query or url.fragment:
                raise ValueError('Use token-free HTTPS or Git SSH; credential URLs, queries and fragments are rejected.')
            if (url.scheme=='https' and url.username is not None) or (url.scheme=='ssh' and url.username not in {None,'git'}):
                raise ValueError('Do not put credentials/user tokens in repository URLs; use an external credential helper or SSH agent.')
            host=url.hostname;path=url.path.lstrip('/')
            if url.port not in {None,22,443}:raise ValueError('Unsupported repository port.')
        path=path.rstrip('/').removesuffix('.git')
        if not path or len(path.split('/'))<2 or '..' in path.split('/'):
            raise ValueError('Configure REPO_URL with OWNER/REPO; no local or credential-bearing URL is accepted.')
        if host.lower()=='github.com':path=path.lower()
        return host.lower()+'/'+path
    expected=identity(repo_url)
    if not isinstance(repo_ref,str) or not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_./-]*',repo_ref) or '..' in repo_ref:
        raise ValueError('REPO_REF must be a branch, tag or full commit SHA, not a command or revision expression.')
    destination=Path(project_dir).expanduser().resolve()
    env=dict(os.environ);env.update(git_env or {});env['GIT_TERMINAL_PROMPT']='0'
    def git(arguments, *, cwd=None, purpose='repository verification'):
        try:result=subprocess.run(['git',*arguments],cwd=cwd,env=env,capture_output=True,text=True,timeout=180)
        except FileNotFoundError:raise RuntimeError('Git is missing; install Git before repository setup.') from None
        except (OSError,subprocess.TimeoutExpired):raise RuntimeError('Git '+purpose+' could not complete; no credentials/output were logged.') from None
        if result.returncode:
            error=result.stderr.lower()
            if any(s in error for s in ('authentication','could not read username','publickey','permission denied')):
                raise RuntimeError('Git authentication failed. Configure an external credential helper or SSH agent with read access; do not put secrets in this notebook.')
            raise RuntimeError('Git '+purpose+' failed. Check repository existence, access and REPO_REF. Raw output is suppressed to protect credentials.')
        return result.stdout.strip()
    def head(path):return git(['rev-parse','--verify','HEAD^{commit}'],cwd=path)
    def resolved_ref(path):
        if re.fullmatch(r'[0-9a-fA-F]{40}|[0-9a-fA-F]{64}',repo_ref):wanted=repo_ref.lower()
        else:
            refs=[repo_ref,repo_ref+'^{}'] if repo_ref.startswith('refs/') else ['refs/heads/'+repo_ref,'refs/tags/'+repo_ref,'refs/tags/'+repo_ref+'^{}']
            lines=git(['ls-remote','--exit-code','origin',*refs],cwd=path,purpose='remote reference lookup')
            mapping=dict((ref,sha) for sha,ref in (line.split() for line in lines.splitlines()))
            if repo_ref.startswith('refs/'):wanted=mapping.get(repo_ref+'^{}',mapping.get(repo_ref))
            else:
                branch=mapping.get('refs/heads/'+repo_ref);tag=mapping.get('refs/tags/'+repo_ref+'^{}',mapping.get('refs/tags/'+repo_ref))
                if branch and tag and branch!=tag:raise ValueError('Branch/tag name is ambiguous; use refs/heads/... or refs/tags/....')
                wanted=branch or tag
            if not wanted:raise ValueError('Configured REPO_REF was not found on origin.')
        return wanted
    existed=destination.exists()
    if existed:
        if not destination.is_dir() or not (destination/'.git').exists():
            raise ValueError('PROJECT_DIR exists but is not a Git checkout; choose a separate empty directory. Nothing was overwritten.')
        top=Path(git(['rev-parse','--show-toplevel'],cwd=destination)).resolve()
        if top!=destination:raise ValueError('PROJECT_DIR must be the repository root, not a nested directory.')
        origin=git(['config','--get','remote.origin.url'],cwd=destination)
        if identity(origin)!=expected:raise ValueError('Existing checkout origin differs from REPO_URL. No files or remotes were changed.')
        if git(['status','--porcelain','--untracked-files=normal'],cwd=destination):
            raise ValueError('Existing checkout has local modifications/untracked files. Commit them yourself or choose a new PROJECT_DIR; no reset/checkout was performed.')
        commit=head(destination)
        wanted=resolved_ref(destination)
        if commit!=wanted:
            raise ValueError('Existing checkout is at '+commit+' but REPO_REF resolves to '+wanted+'. Choose a new PROJECT_DIR or update it explicitly; nothing was reset.')
    else:
        destination.parent.mkdir(parents=True,exist_ok=True)
        # A failed clone stays in its own scratch directory, not PROJECT_DIR.
        with tempfile.TemporaryDirectory(prefix='.pdeep-clone-',dir=destination.parent) as scratch:
            checkout=Path(scratch)/'checkout'
            git(['clone','--no-checkout','--',repo_url,str(checkout)],purpose='clone')
            wanted=resolved_ref(checkout)
            git(['fetch','--depth','1','origin',wanted],cwd=checkout,purpose='configured reference fetch')
            git(['checkout','--detach','FETCH_HEAD'],cwd=checkout,purpose='fresh checkout')
            commit=head(checkout)
            if commit!=wanted:raise RuntimeError('Fetched checkout differs from the resolved reference; setup stopped.')
            required=('pyproject.toml','src/pdeep/__init__.py','src/pdeep/colab_setup.py','docs/scientific_contract_v1.md')
            if any(not (checkout/p).is_file() for p in required):
                raise ValueError('Repository/ref lacks the expected PDEep structure or this Colab setup version; select a compatible commit.')
            os.rename(checkout,destination)
    required=('pyproject.toml','src/pdeep/__init__.py','src/pdeep/colab_setup.py','docs/scientific_contract_v1.md')
    if any(not (destination/p).is_file() for p in required):raise ValueError('Expected pyproject.toml, pdeep package/setup and scientific contract are missing.')
    return {'project_dir':str(destination),'repository_identity':expected,'requested_ref':repo_ref,'git_commit':commit,
            'existing_checkout':existed,'worktree_modified_by_setup':not existed,'credentials_recorded':False}

BOOTSTRAP = None
if IS_COLAB:
    print("1. Verifying repository and preparing dependencies before scientific project imports.")
    BOOTSTRAP = ensure_checkout(REPO_URL, REPO_REF, PROJECT_DIR)
    import runpy
    _reference_setup = runpy.run_path(str(Path(BOOTSTRAP["project_dir"]) / "src/pdeep/smooth/bootstrap.py"))
    ENVIRONMENT_SETUP = _reference_setup["prepare_reference_environment"](BOOTSTRAP["project_dir"], is_colab=True)
    _reference_setup["install_project"](BOOTSTRAP["project_dir"], source_revision=BOOTSTRAP["git_commit"])
    print("Dependency preparation complete; no loaded binary was replaced. Existing numerical certification is resolved below; no automatic requalification.")
print("2. Scientific imports and resolved environment.")
from pdeep.colab_setup import finish_setup
SETUP = finish_setup(is_colab=IS_COLAB, bootstrap=BOOTSTRAP, use_drive=USE_GOOGLE_DRIVE,
    drive_root=DRIVE_ROOT, local_output_dir=LOCAL_OUTPUT_DIR)
SETUP_CONFIGURATION = (REPO_URL, REPO_REF, PROJECT_DIR, USE_GOOGLE_DRIVE, DRIVE_ROOT, LOCAL_OUTPUT_DIR)
ROOT = Path(SETUP["project_dir"])
from IPython.display import display, Markdown
from pdeep.notebook_workflows import initialize
from pdeep.runtime import capabilities
print("Colab:", IS_COLAB, "| imported source:", ROOT / "src/pdeep")
print("Exact Git commit:", SETUP["git_commit"] or "unavailable: local project is not a Git checkout")
print("Python:", SETUP["runtime"]["python"])
print("Dependency versions:", json.dumps(SETUP["runtime"]["dependencies"], indent=2))
for c in capabilities(): print(c.device, "available="+str(c.available), c.detail)
if IS_COLAB: print("Validated CUDA:", SETUP["cuda"])
print("Local unified root:", Path(SETUP["local_output_dir"]) / "G1_unified")
print("Drive unified root:", None if not SETUP["drive_root"] else Path(SETUP["drive_root"]) / "G1_unified")
print(SETUP["message"])

from pdeep.smooth.bootstrap import environment_versions
print("Python executable:", sys.executable)
print("Installed / loaded dependencies:", json.dumps(environment_versions(), sort_keys=True))

assert all((ROOT / p).is_file() for p in ("src/pdeep/unified/workflow.py", "docs/g1_unified_experiment_v1.md", "notebooks/04_g1_unified.ipynb")), "Configured repository/ref lacks the G1 unified workflow; select its reviewed commit."


In [ ]:
from pdeep.learning_diagnostics.certificates import resolve_certification, copy_certificate, SUPPLIED_ID
CERTIFICATE_ID = SUPPLIED_ID
CERTIFICATE_PATH = None  # Optional full issued JSON path; no upload-path assumption.
USE_EXISTING_LOCAL_QUALIFICATION = not IS_COLAB  # Explicit local policy: reuse existing, never recompute.
CERTIFICATE_CACHE = Path(SETUP["local_output_dir"]) / "certifications/G1_smooth_v1"

ISSUED_CERTIFICATION = resolve_certification("G1_smooth_v1", {"certification_id": CERTIFICATE_ID},
    configured_path=CERTIFICATE_PATH, drive_root=SETUP["drive_root"], cache_dir=CERTIFICATE_CACHE)
display(ISSUED_CERTIFICATION.summary())
CERTIFICATION = ISSUED_CERTIFICATION
if not IS_COLAB and USE_EXISTING_LOCAL_QUALIFICATION and not CERTIFICATION.verified:
    CERTIFICATION = resolve_certification("G1_smooth_v1", {"certification_id": None}, cache_dir=CERTIFICATE_CACHE)
    print("Explicit local policy: selecting an existing native-runtime qualification; issued Linux qualification remains incompatible here.")
    display(CERTIFICATION.summary())
CERTIFICATE = CERTIFICATION.record if CERTIFICATION.verified else None
if CERTIFICATE is None:
    print("Training blocked. Restore the certified runtime/settings or place the full issued JSON in the displayed cache/Drive directory. No requalification has been executed.")
if CERTIFICATION.verified and SETUP["drive_root"]:
    try:
        CERTIFICATE_DRIVE_PATH = copy_certificate(CERTIFICATION.location,
            Path(SETUP["drive_root"]) / "certifications/G1_smooth_v1")
        print("Full certificate byte-copy verified:", CERTIFICATE_DRIVE_PATH)
    except Exception as error:
        print("CERTIFICATE DRIVE COPY FAILED; verified local copy retained:", type(error).__name__, str(error))
PERSISTENCE = SETUP["persistence"]

## 2. Shared experiment configuration
The numerical certificate is finite evidence for 96 cases. Newly generated cases need their own QC; the dataset adds a separate stratified finer-reference qualification. All-eight training is a versioned experiment extension, with the existing prior, split keys and frozen TRAIN normalizers.

Three independent networks; matched physical cases, observation/query streams and task order. C variants start with identical C weights. Auxiliary source/initial fields are **supervised targets only**. `auto` AMP resolves explicitly to FP32; the T4 has no assumed BF16 support. FP16/GradScaler with coordinate derivatives is not enabled without numerical verification.

For a bounded user-run smoke set `G1_MODE="smoke"`; it resolves 2 cases/task/split, 3 planned updates and a 2-update session. Pilot/research retain a declared 30,000-update schedule.

In [ ]:
from pdeep.unified.config import DatasetConfig
from pdeep.unified.workflow import configurations, preview
G1_MODE = "research"
G1_RUN_MODE = "preview"
G1_MODELS = ["A_GEO/KIN_001", "C_GEO/FIELD_ONLY", "C_GEO/ALL_001"]
G1_DATASET_PRESET = "research"
G1_MODEL_CAPACITY = "medium"
G1_PDES = ["poisson", "heat", "wave"]
G1_GEOMETRIES = ["square", "rectangle", "lshape", "disk", "ellipse", "hole", "annulus", "polygon"]
G1_CASES_PER_UPDATE = 4
G1_QUERIES_PER_CASE = 4096
G1_QUERY_CHUNK_SIZE = 2048
G1_OBSERVATION_COUNTS = [8, 16, 32, 64, 128]
G1_TARGET_SUCCESSFUL_UPDATES = 30000
G1_STOP_AFTER_UPDATES = 3000
G1_LR = 3e-4
G1_WEIGHT_DECAY = 1e-4
G1_GRAD_CLIP = 1.0
G1_AMP = "auto"
G1_VALIDATE_EVERY = 500
G1_CHECKPOINT_EVERY = 500
G1_SEED = 42
G1_NEURAL_SEED = None  # Optional separate initialization seed; data streams stay fixed.
G1_CHECKPOINT_LOOKUP = "none"
G1_ACTION = "fresh"
G1_EXPERIMENT_ID = None  # Fresh UUID. For exact resume set the ORIGINAL displayed experiment ID.
G1_RESUME_PATHS = {}  # Exact manifest paths keyed by model, or load a known experiment.json below.
G1_DATA_ACTION = "preview"  # preview | prepare | load | restore
G1_DATASET_MANIFEST = None  # Exact local index (load) or known Drive index (restore).
G1_RUN_PREFLIGHT = False
G1_EVALUATE = False
G1_EVAL_SPLIT = "validation"  # Explicitly choose test only after all tuning/checkpoint selection.
G1_EVAL_SELECTION = "best"

assert G1_CASES_PER_UPDATE == 4 and G1_OBSERVATION_COUNTS == [8,16,32,64,128]
if G1_MODE == "smoke":
    G1_DATASET_PRESET = "smoke"
    G1_TARGET_SUCCESSFUL_UPDATES, G1_STOP_AFTER_UPDATES = 3, 2
    G1_QUERIES_PER_CASE, G1_QUERY_CHUNK_SIZE = 128, 64
G1_DATA_CONFIG = DatasetConfig(preset=G1_DATASET_PRESET, pdes=tuple(G1_PDES), families=tuple(G1_GEOMETRIES), seed=G1_SEED,
    stream_updates=G1_TARGET_SUCCESSFUL_UPDATES, stream_queries=G1_QUERIES_PER_CASE)
G1_ROOT = Path(SETUP["local_output_dir"]) / "G1_unified"
G1_CONFIGS = configurations(G1_DATA_CONFIG, models=G1_MODELS, experiment_id=G1_EXPERIMENT_ID,
    device=SETUP["training_device"], capacity=G1_MODEL_CAPACITY, updates=G1_TARGET_SUCCESSFUL_UPDATES,
    query_count=G1_QUERIES_PER_CASE, query_chunk=G1_QUERY_CHUNK_SIZE, seed=G1_SEED,
    learning_rate=G1_LR, weight_decay=G1_WEIGHT_DECAY, grad_clip=G1_GRAD_CLIP,
    validate_every=G1_VALIDATE_EVERY, checkpoint_every=G1_CHECKPOINT_EVERY, amp=G1_AMP, mode=G1_MODE, neural_seed=G1_NEURAL_SEED)
G1_EXPERIMENT_ID = G1_CONFIGS[0].experiment_id
G1_PREFLIGHT_PATH = None
G1_DATASET = None
G1_RESULT = None
display(preview(G1_DATA_CONFIG, G1_CONFIGS, G1_ROOT))
print("Experiment ID:", G1_EXPERIMENT_ID)
print("Session stop:", G1_STOP_AFTER_UPDATES, "on the declared", G1_TARGET_SUCCESSFUL_UPDATES, "update schedule")

## 3. Dataset inspection and generation
Set `G1_DATA_ACTION="prepare"` deliberately, then rerun configuration and this cell. This can be CPU/storage expensive. Each immutable completed case/draw record is reused after interruption; failures are quarantined, never substituted with easier cases. Each physical case is propagated once offline with the original FULL CPU64 reference. Labels for every independent counter-keyed encounter in the declared schedule are prepared and the full trajectory is discarded. Online regenerates those exact continuous coordinates and fresh per-encounter noise: no recycled sensor/query bank and no PDE solves inside GPU updates.

The first TRAIN case/task receives independent P3 h=.01/dt=.001 and fixed-space dt-halving checks. This is **separate finite dataset qualification**, not a universal guarantee from the old certificate. All required checks must pass before training.

`load` uses the exact local index. `restore` uses the exact Drive index and its verified explicit artifact locations; there is no recursive search. Actual shared-cache storage/CPU costs are reported during preparation; a large fallback can require substantial disk.

In [ ]:
from pdeep.unified.workflow import prepare
if G1_DATA_ACTION != "preview":
    CERTIFICATION.require()
    G1_DATASET = prepare(G1_DATA_CONFIG, G1_ROOT, CERTIFICATE, action=G1_DATA_ACTION,
        manifest=G1_DATASET_MANIFEST, persistence=PERSISTENCE)
    G1_DATASET_MANIFEST = str(G1_DATASET.path)
    display(G1_DATASET.summary())
else:
    print("Dataset preview only: no reference preparation, dataset loading or qualification executed.")

## 4. Model inventory and bounded preflight
Preflight performs real forward/backward passes on isolated models, **zero optimizer updates**, all three PDE objectives at the configured query count, derivative reachability, and measured peak allocated/reserved CUDA memory with an Adam-state allowance. Four independent cases are accumulated one at a time. An out-of-budget setting fails with an explicit suggestion; batch size and losses are not silently reduced.

Set `G1_RUN_PREFLIGHT=True` after the dataset is prepared/restored. Repeat after changing configuration or restarting on different hardware.

In [ ]:
from pdeep.unified.preflight import inventory, run_preflight
if CERTIFICATE is not None:
    from pdeep.smooth.certification import normalizers
    display(inventory(G1_CONFIGS, normalizers(CERTIFICATE)))
if G1_RUN_PREFLIGHT:
    assert G1_DATASET is not None, "Prepare/load the dataset first."
    G1_PREFLIGHT_PATH, G1_PREFLIGHT = run_preflight(G1_CONFIGS, G1_DATASET, G1_ROOT)
    display({"path": G1_PREFLIGHT_PATH, "passed": G1_PREFLIGHT["passed"], "seconds": G1_PREFLIGHT["seconds"], "optimizer_steps": 0})
else:
    print("Preflight disabled. Research training remains gated.")

## 5. Research training — one synchronous sequential call
Set `G1_RUN_MODE="train"`, rerun configuration, load the prepared dataset, rerun preflight, then execute this cell. It runs the selected three models in recorded order. Default: stop each model at 3,000 **successful** updates on its unchanged 30,000-update schedule.

To continue, keep the original experiment ID and all scientific settings, select `G1_ACTION="resume"`, `G1_CHECKPOINT_LOOKUP="explicit"`, provide exact verified paths in `G1_RESUME_PATHS`, and set stop to 30,000. A finished model is retained, a partial model resumes, and a model without a supplied path starts fresh. Legacy single-case checkpoints are rejected. No manual configuration between model runs.

In [ ]:
from pdeep.unified.training import train_models
G1_RESULT = train_models(G1_CONFIGS, G1_DATASET, G1_ROOT, run_mode=G1_RUN_MODE,
    stop_after=G1_STOP_AFTER_UPDATES, lookup=G1_CHECKPOINT_LOOKUP, action=G1_ACTION,
    resume_paths=G1_RESUME_PATHS, persistence=PERSISTENCE, preflight_path=G1_PREFLIGHT_PATH)
if G1_RUN_MODE == "preview":
    display(G1_RESULT)
else:
    display({key: {k:r[k] for k in ("status","successful","latest","best")} for key,r in G1_RESULT.items()})

## 6. Independent validation/test evaluation
Evaluation loads only known artifacts, never optimizes. `best` uses the stored validation-selected weights, with equal PDE and active-quantity weighting. Test is a deliberate final action; never use it to choose weights or checkpoints.

Outputs include per-case and 24-task statistics, paired intervals, nine physical times, weak/initial/boundary and auxiliary errors, wave interior kinematic/dynamic diagnostics, sensor-count/noise/value/position ablations, zero/IDW baselines and an optional geometry-specific C3 Gaussian inverse baseline when its separately prepared sensitivity assets are available (otherwise explicitly N/A). The Gaussian baseline is exact finite-dimensional conditioning of the bounded cached P4 operator, **not an exact continuum posterior or T1 D**.

In [ ]:
from pdeep.unified.workflow import evaluate_checkpoints
G1_EVALUATION = None
if G1_EVALUATE:
    assert G1_DATASET is not None, "Load the prepared dataset first."
    G1_EVAL_PATHS = dict(G1_RESUME_PATHS)
    if G1_RUN_MODE == "train":
        G1_EVAL_PATHS.update({k:r["latest"] for k,r in G1_RESULT.items()})
    G1_EVALUATION = evaluate_checkpoints(G1_EVAL_PATHS, G1_CONFIGS, G1_DATASET, G1_ROOT,
        split=G1_EVAL_SPLIT, selection=G1_EVAL_SELECTION)
    display({"directory":G1_EVALUATION["directory"], "models":list(G1_EVALUATION["report"]["models"])})
else:
    print("Evaluation disabled; no test data inspected.")

## 7. Compact visualizations and comparison
A few aggregate figures cover task errors, information sensitivity, time, consistency and cost. Dataset/source/sensor examples and a detailed single-case inspector are optional reusable functions in `pdeep.unified.plots`; they never trigger a solve or training. Familiar fixed instances with unseen fields do not establish unseen-instance/family transfer. `DatasetConfig(holdout_family=...)` declares a separate future leave-family-out experiment; additional instances still require independent qualification.

In [ ]:
from IPython.display import Image
if G1_EVALUATION:
    for path in G1_EVALUATION["plots"]:
        display(Image(filename=path))
else:
    print("No trained-model comparison is claimed before user execution.")

## 8. Checkpoints, exports and disconnect recovery
Local checkpoints commit and verify first; periodic verified copies go under `DriveRoot/G1_unified/runs/<experiment>/<model>/checkpoints/<run_id>/<artifact_id>/`. Dataset index: `DriveRoot/G1_unified/datasets/<dataset_id>/index.json`. The index records exact verified cache artifact locations. A Drive failure preserves local artifacts and warns prominently; Colab local storage is temporary.

After disconnect: rerun setup, select the original dataset/configuration/experiment ID, restore the exact dataset index, supply exact model manifests (or use `explicit_experiment_paths` on the known saved `experiment.json`), rerun preflight, and resume. Model, Adam moments, full scheduler, RNG, keyed order, counters and best weights restore together. Source/Torch/device/config incompatibilities fail explicitly. Optional exports use the known checkpoint, never discovery. Repeat neural seeds as separate explicit experiments after the first successful research run.

In [ ]:
if G1_RUN_MODE == "train" and G1_RESULT:
    display({key:{"latest":r["latest"], "best":r["best"], "local_and_Drive":r["persistence"]} for key,r in G1_RESULT.items()})
else:
    print("No checkpoints written by preview. Original experiments and checkpoints remain untouched.")
print("Dataset index:", G1_DATASET_MANIFEST)
print("Exact Git revision:", SETUP["git_commit"])
print("CUDA/Drive claims require actual Colab execution; local filesystem tests are not Drive certification.")